# Notebook setup

In [ ]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

%load_ext autoreload
%autoreload 2
%matplotlib inline

# Data

In [ ]:
from src.config import Configuration
import json

CONFIG = Configuration()

train_df = pd.read_csv(CONFIG.train_data)
train_df_train = train_df.sample(frac=1-CONFIG.val_split, random_state=42)
train_df_val = train_df.drop(train_df_train.index)

test_df = pd.read_csv(CONFIG.test_data)

print(f"Train shape: {train_df_train.shape}")
print(f"Validation shape: {train_df_val.shape}")
print(f"Test shape: {test_df.shape}")

In [ ]:
import ast
csv_path = os.path.join(CONFIG.LOGS_FOLDER, "dataset_test_original.csv")
predictions_df = pd.read_csv(csv_path)

predictions_df["genre"] = predictions_df["genre"].apply(ast.literal_eval)
predictions_df["genre"].to_list()

---

In [ ]:
csv_path = os.path.join(CONFIG.LOGS_FOLDER, "gemini_2.5-pro.csv")
gemini = pd.read_csv(csv_path)

In [53]:
csv_path = os.path.join(CONFIG.DATA_FOLDER, "dataset_test.csv")
test_df = pd.read_csv(csv_path)

In [ ]:
[movie for movie in gemini['movie_name'].to_list() if movie not in test['movie_name'].to_list()]

In [ ]:
len(test[['movie_name', "description"]].drop_duplicates())

---

In [ ]:
test[test["movie_name"].isin(["The Enforcer", "Close", "Point Break","The Gift"])]

In [50]:
import ast
csv_path = os.path.join(CONFIG.LOGS_FOLDER, "gemini_prediction.csv")
gemini = pd.read_csv(csv_path)

gemini["genre"] = gemini["genre"].apply(ast.literal_eval)
gemini["genre"] = [", ".join(pred) if pred else "" for pred in gemini["genre"].to_list()]
gemini["description"] = test_df["description"].values

gemini = gemini[["movie_name", "description", "genre"]]
gemini.to_csv(os.path.join(CONFIG.LOGS_FOLDER, "gemini_final_submission.csv"), index=False)

In [58]:
for test, gem in zip(test_df['description'].to_list(), gemini['description'].to_list()):
    if test != gem:
        print(f"Mismatch: {test} != {gem}")